In [1]:
%pip install pandas

  Using cached pandas-3.0.3-cp314-cp314-win_amd64.whl.metadata (19 kB)
Using cached pandas-3.0.3-cp314-cp314-win_amd64.whl (9.9 MB)
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install requests


  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached charset_normalizer-3.4.7-cp314-cp314-win_amd64.whl.metadata (41 kB)
  Using cached idna-3.18-py3-none-any.whl.metadata (6.1 kB)
  Using cached urllib3-2.7.0-py3-none-any.whl.metadata (6.9 kB)
Using cached requests-2.34.2-py3-none-any.whl (73 kB)
Using cached charset_normalizer-3.4.7-cp314-cp314-win_amd64.whl (159 kB)
Using cached idna-3.18-py3-none-any.whl (65 kB)
Using cached urllib3-2.7.0-py3-none-any.whl (131 kB)

   ---------------------------------------- 0/5 [urllib3]
   ---------------------------------------- 0/5 [urllib3]
   ---------------------------------------- 0/5 [urllib3]
   ---------------------------------------- 0/5 [urllib3]
   ---------------------------------------- 0/5 [urllib3]
   ---------------------------------------- 0/5 [urllib3]
   ---------------------------------------- 0/5 [urllib3]
   ---------------------------------------- 0/5 [urllib3]
   -------------------------------

In [3]:
import requests
import pandas as pd

In [4]:
BASE = "https://pokeapi.co/api/v2/"

N = 151

LEGENDARY = [144, 145, 146, 150, 151]

print("Listo")

Listo


In [5]:
r =  requests.get(f"{BASE}pokemon/25", timeout=30)
r.raise_for_status()
data = r.json()

print("name:", data["name"])
print("height:", data["height"], "| weight:", data["weight"])
print("claves:", list(data.keys())[:10])

name: pikachu
height: 4 | weight: 60
claves: ['abilities', 'base_experience', 'cries', 'forms', 'game_indices', 'height', 'held_items', 'id', 'is_default', 'location_area_encounters']


In [7]:
print("types:", data["types"])
print()
print("stats[0]:", data["stats"][0])

types: [{'slot': 1, 'type': {'name': 'electric', 'url': 'https://pokeapi.co/api/v2/type/13/'}}]

stats[0]: {'base_stat': 35, 'effort': 0, 'stat': {'name': 'hp', 'url': 'https://pokeapi.co/api/v2/stat/1/'}}


In [13]:
def flatten (data):
    """JSON de un Pokémon -> fila plana (dict) lista para el CSV."""

    stats = { s["stat"]["name"].replace("-", "_"): s["base_stat"] for s in data["stats"] }
    types = [ t["type"]["name"] for t in data["types"]]
    sprite = (data["sprites"]["other"]["official-artwork"]["front_default"] or data["sprites"]["front_default"])

    return {
        "id": data["id"],
        "name": data["name"].capitalize(),
        "type_1": types[0],
        "type_2": types[1] if len(types) > 1 else None,
        **stats,
        "total": sum(stats.values()),
        "height_m": data["height"] / 10,
        "weight_kg": data["weight"] / 10,
        "base_exp": data.get("base_experience"),
        "legendary": data["id"] in LEGENDARY,
        "sprite": sprite,
    }

#probamos con pikachu
flatten(data)
 

{'id': 25,
 'name': 'Pikachu',
 'type_1': 'electric',
 'type_2': None,
 'hp': 35,
 'attack': 55,
 'defense': 40,
 'special_attack': 50,
 'special_defense': 50,
 'speed': 90,
 'total': 320,
 'height_m': 0.4,
 'weight_kg': 6.0,
 'base_exp': 112,
 'legendary': False,
 'sprite': 'https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/other/official-artwork/25.png'}

In [14]:
rows = []
for i in range(1, N+1):
    r =  requests.get(f"{BASE}pokemon/{i}", timeout=30)
    r.raise_for_status()
    rows.append(flatten(r.json()))
    print(f" {i}/{N} {rows[-1]['name']}", end="\r")


df = pd.DataFrame(rows)
print("\nForma:", df.shape)
df.head()

 151/151 Mewtwoitel
Forma: (151, 16)


,id,name,type_1,type_2,hp,attack,defense,special_attack,special_defense,speed,total,height_m,weight_kg,base_exp,legendary,sprite
0,1,Bulbasaur,grass,poison,45,49,49,65,65,45,318,0.7,6.9,64,False,https://raw.githubusercontent.com/PokeAPI/spri...
1,2,Ivysaur,grass,poison,60,62,63,80,80,60,405,1.0,13.0,142,False,https://raw.githubusercontent.com/PokeAPI/spri...
2,3,Venusaur,grass,poison,80,82,83,100,100,80,525,2.0,100.0,236,False,https://raw.githubusercontent.com/PokeAPI/spri...
3,4,Charmander,fire,NaN,39,52,43,60,50,65,309,0.6,8.5,62,False,https://raw.githubusercontent.com/PokeAPI/spri...
4,5,Charmeleon,fire,NaN,58,64,58,80,65,80,405,1.1,19.0,142,False,https://raw.githubusercontent.com/PokeAPI/spri...


In [15]:
df.to_csv("pokemon.csv", index=False) #columna de indice
print("Archivo pokemon.csv creado con éxito")

Archivo pokemon.csv creado con éxito
